In [1]:
import os
import json

from dotenv import load_dotenv
from pathlib import Path
from mistralai import Mistral
from tqdm import tqdm

from constants import AnnotationsReduced
from model_utils import (
    get_binary_classification_fields,
    get_classification_fields,
    get_multiple_choice_fields,
    get_regression_fields,
    create_label_to_id_map,
    labels_to_bits
)

In [2]:
# Set the API key for Mistral
load_dotenv()  # Load environment variables from .env file
mistral_api_key = os.getenv("MISTRAL_API_KEY")

# Initialize the Mistral client
client = Mistral(api_key=mistral_api_key)

In [3]:
# Parameters
TRAIN_FILE_NAME = "train_mistral.jsonl"
VALIDATION_FILE_NAME = "validation_mistral.jsonl"
TEST_FILE_NAME = "test_mistral.jsonl"

#TRAINED_MODEL_ID = 'ft:classifier:ministral-3b-latest:6f88df17:20260106:037d39cd' # 10 epochs
TRAINED_MODEL_ID = 'ft:classifier:ministral-3b-latest:6f88df17:20260106:0ff0318a' # 20 epochs
#TRAINED_MODEL_ID = 'ft:classifier:ministral-3b-latest:6f88df17:20260106:23a3cc29' # 30 epochs

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
    'test': TEST_FILE_NAME,
}

paths = {
    split: Path('../MISTRAL/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    # Load the samples
    with open(path, "r") as f:
        data[split] = [json.loads(l) for l in f.readlines()]

train_data, validation_data, test_data = data['train'], data['validation'], data['test']

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")

len(train_data) = 166
len(validation_data) = 54
len(test_data) = 55


In [5]:
reg_fields = get_regression_fields(AnnotationsReduced)
cl_fields = get_classification_fields(AnnotationsReduced)
mc_fields = get_multiple_choice_fields(AnnotationsReduced)
bc_fields = get_binary_classification_fields(AnnotationsReduced)
label_to_id_map = create_label_to_id_map(AnnotationsReduced)

In [6]:
def save_actual_labels(results_dict: dict, split: str, labels: dict, label_to_id_map: dict) -> None:
    for field in bc_fields:
        positive_label = str(label_to_id_map[field]['id_to_label'][1])
        actual_label = str(labels[field])
        index = int(actual_label == positive_label)
        results_dict[split]['actual'][field].append(index)
    for field in cl_fields:
        actual_label = labels[field]
        index = label_to_id_map[field]['label_to_id'][actual_label]
        results_dict[split]['actual'][field].append(index)
    for field in mc_fields:
        l = labels[field]
        bits = labels_to_bits(l, label_to_id_map[field]['label_to_id'])
        results_dict[split]['actual'][field].append(bits)

def save_classifier_response(results_dict: dict, split: str, response, label_to_id_map: dict) -> None:
    for field in bc_fields:
        scores = response.results[0][field].scores
        positive_label = str(label_to_id_map[field]['id_to_label'][1])
        score = scores[positive_label]
        results_dict[split]['predicted'][field].append(score)
    for field in cl_fields + mc_fields:
        scores = response.results[0][field].scores
        l = []
        for i in range(len(scores)):
            label = label_to_id_map[field]['id_to_label'][i]
            score = scores[label]
            l.append(score)
        results_dict[split]['predicted'][field].append(l)

In [7]:
# Create save dict
results = {
    'validation': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'test': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'info':{
        'regression_fields': reg_fields,
        'classification_fields': cl_fields,
        'binary_classification_fields': bc_fields,
        'multiple_choice_fields': mc_fields,
        'normalization_stats': {},
        'label_to_id_map': label_to_id_map,
    }
}
# Save predictions
# Validation
for sample in tqdm(validation_data):
    # Save actual labels
    save_actual_labels(results,
                       'validation',
                       sample['labels'],
                       label_to_id_map)
    # Save prediction
    classifier_response = client.classifiers.classify(
        model=TRAINED_MODEL_ID,
        inputs=[sample['text']],
    )
    save_classifier_response(results,
                             'validation',
                             classifier_response,
                             label_to_id_map)
# Test
for sample in tqdm(test_data):
    # Save actual labels
    save_actual_labels(results,
                       'test',
                       sample['labels'],
                       label_to_id_map)
    # Save prediction
    classifier_response = client.classifiers.classify(
        model=TRAINED_MODEL_ID,
        inputs=[sample['text']],
    )
    save_classifier_response(results,
                             'test',
                             classifier_response,
                             label_to_id_map)

100%|██████████| 55/55 [00:16<00:00,  3.36it/s]


In [8]:
##############
# Save results
##############
with open('results_mistral_20.json', "w") as f:
    json.dump(results, f, indent=4)
print(f"Results saved")

Results saved
